# 7CS537 CW1, Notebook 02: Model Training and Evaluation (CICIDS2017)

**Purpose:** train, test and compare AI/ML models for detecting malicious network flows at TeleComNet Ltd, using the cleaned dataset from Notebook 01.

**Main task:** binary classification, BENIGN (0) versus attack (1). This mirrors the SOC decision of whether a flow should raise an alert.

**Models compared:**
| Model | Type | Why it is included |
|---|---|---|
| Logistic Regression | Supervised, linear | Simple, interpretable baseline. It shows how much the more complex models actually add. |
| Random Forest | Supervised, bagged trees | Handles mixed feature scales and non-linear patterns. Gives feature importance for explainability. |
| XGBoost | Supervised, boosted trees | Usually the strongest model on tabular data, with built-in imbalance weighting. |
| Isolation Forest | Unsupervised anomaly detection | Learns only from benign traffic, so it can in principle flag attack types never seen in training. |

**Methodology summary:**
1. Stratified 70/15/15 train, validation and test split by the original label, so even the rarest attacks appear in every part.
2. Validation data is used only for early stopping and choosing thresholds. The test set is used once, for the final results.
3. Class imbalance is handled with class weights, not synthetic oversampling.
4. Extra experiments: threshold trade-off, the Destination Port ablation, held-out "unseen attack" families, and a multi-class extension.

**Outputs saved to `outputs/`** with the prefix `02_`. Run the cells in order. The longer experiments can be switched off in Section 1.

## 1. Setup and configuration

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score, matthews_corrcoef,
                             roc_curve, precision_recall_curve, classification_report)
import sklearn
import xgboost as xgb
from xgboost import XGBClassifier

try:
    from IPython.display import display
except ImportError:
    display = print

OUT_DIR = "outputs"
MODEL_DIR = "models"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

RANDOM_STATE = 42
N_JOBS = -1                 # use all CPU cores

# Set to a fraction (e.g. 0.3) for a quick trial run. None uses all rows; report results from None.
SAMPLE_FRAC = None

# Optional experiments (each adds several minutes)
RUN_PORT_ABLATION = True
RUN_UNSEEN_ATTACKS = True
RUN_MULTICLASS = True

print("scikit-learn", sklearn.__version__, "| xgboost", xgb.__version__, "| pandas", pd.__version__)

## 2. Load the cleaned dataset from Notebook 01

In [ ]:
if os.path.exists("data/cicids2017_clean.parquet"):
    df = pd.read_parquet("data/cicids2017_clean.parquet")
else:
    df = pd.read_pickle("data/cicids2017_clean.pkl")

with open(f"{OUT_DIR}/01_feature_list.txt") as fh:
    feature_cols = [l.strip() for l in fh if l.strip()]
PORT_COL = "Destination Port"

if SAMPLE_FRAC:
    small = df["Label"].map(df["Label"].value_counts()) <= 50      # keep every row of the rarest classes
    df = pd.concat([df[small],
                    df[~small].groupby("Label", observed=True).sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)],
                   ignore_index=True)
    print(f"Quick-trial mode: using {SAMPLE_FRAC:.0%} of each large class")

df["Label"] = df["Label"].astype(str)
df["attack_family"] = df["attack_family"].astype(str)
print(f"{len(df):,} rows, {len(feature_cols)} features")
print(df["is_attack"].value_counts().rename({0: "Benign", 1: "Attack"}))

## 3. Train, validation and test split
The split is stratified by the original 15-class label rather than the binary target. With a binary stratification, a class as small as Heartbleed (11 rows) could end up entirely in training, and its detection could never be tested.

In [ ]:
idx = np.arange(len(df))
idx_train, idx_temp = train_test_split(idx, test_size=0.30, stratify=df["Label"], random_state=RANDOM_STATE)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, stratify=df["Label"].iloc[idx_temp],
                                     random_state=RANDOM_STATE)

X_all = df[feature_cols].to_numpy(dtype=np.float32)
y_all = df["is_attack"].to_numpy()
X_train, y_train = X_all[idx_train], y_all[idx_train]
X_val, y_val = X_all[idx_val], y_all[idx_val]
X_test, y_test = X_all[idx_test], y_all[idx_test]
labels_test = df["Label"].to_numpy()[idx_test]

split_table = pd.DataFrame({
    "train": df["Label"].iloc[idx_train].value_counts(),
    "validation": df["Label"].iloc[idx_val].value_counts(),
    "test": df["Label"].iloc[idx_test].value_counts(),
}).fillna(0).astype(int).sort_values("train", ascending=False)
split_table.loc["TOTAL"] = split_table.sum()
split_table.to_csv(f"{OUT_DIR}/02_split_counts.csv")
display(split_table)

## 4. Evaluation helpers
Metrics are chosen for a SOC setting rather than accuracy alone. With about 83% benign traffic, a model that labels everything benign would still score about 83% accuracy.
- **Recall (detection rate)** and **false negative rate**: the share of attacks detected and missed.
- **Precision** and **false positive rate**: how trustworthy alerts are, and how much analyst workload false alarms create.
- **F1, MCC and PR-AUC**: single-number summaries that remain informative under class imbalance.
- **False alarms per 100,000 benign flows**: the false positive rate expressed in operational terms.

In [ ]:
results, test_scores, thresholds_used = [], {}, {}

def evaluate(name, y_true, score, threshold, fit_time=None, predict_time=None, store=True):
    y_pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    row = {
        "model": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_detection_rate": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "fpr": fp / (fp + tn) if (fp + tn) else 0.0,
        "fnr": fn / (fn + tp) if (fn + tp) else 0.0,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, score),
        "pr_auc": average_precision_score(y_true, score),
        "false_alarms_per_100k_benign": fp / (fp + tn) * 100_000 if (fp + tn) else 0.0,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "fit_time_s": fit_time,
        "predict_time_s": predict_time,
    }
    if store:
        results.append(row)
        test_scores[name] = score
        thresholds_used[name] = threshold
    return row

def show(row):
    keys = ["accuracy", "precision", "recall_detection_rate", "f1", "fpr", "fnr", "mcc", "roc_auc", "pr_auc"]
    print(f"{row['model']}: " + ", ".join(f"{k}={row[k]:.4f}" for k in keys))
    print(f"   TN={row['TN']:,} FP={row['FP']:,} FN={row['FN']:,} TP={row['TP']:,}")

def timed(fn, *a, **k):
    t = time.time()
    out = fn(*a, **k)
    return out, time.time() - t

## 5. Model 1: Logistic Regression (baseline)
Features are standardised first, because a linear model is sensitive to feature scale (Notebook 01 showed ranges spanning many orders of magnitude). `class_weight="balanced"` gives the minority attack class more weight in training.

In [ ]:
logreg = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
_, t_fit = timed(logreg.fit, X_train, y_train)
s, t_pred = timed(lambda X: logreg.predict_proba(X)[:, 1], X_test)
show(evaluate("Logistic Regression", y_test, s, 0.5, t_fit, t_pred))

## 6. Model 2: Random Forest
`class_weight="balanced_subsample"` reweights classes within each tree's bootstrap sample. Tree models do not need feature scaling.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample", min_samples_leaf=1,
                            n_jobs=N_JOBS, random_state=RANDOM_STATE)
_, t_fit = timed(rf.fit, X_train, y_train)
s, t_pred = timed(lambda X: rf.predict_proba(X)[:, 1], X_test)
show(evaluate("Random Forest", y_test, s, 0.5, t_fit, t_pred))

## 7. Model 3: XGBoost
`scale_pos_weight` is set to the ratio of benign to attack flows in training, which is XGBoost's standard way of handling imbalance. Early stopping on the validation set chooses the number of trees, so the test set is never used for tuning.

In [ ]:
spw = float((y_train == 0).sum() / (y_train == 1).sum())
xgb_params = dict(n_estimators=1000, learning_rate=0.1, max_depth=8, subsample=0.8, colsample_bytree=0.8,
                  tree_method="hist", scale_pos_weight=spw, eval_metric="aucpr",
                  n_jobs=N_JOBS, random_state=RANDOM_STATE)
xgbm = XGBClassifier(early_stopping_rounds=30, **xgb_params)
_, t_fit = timed(xgbm.fit, X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
best_n_trees = xgbm.best_iteration + 1
print(f"scale_pos_weight={spw:.2f}, trees chosen by early stopping: {best_n_trees}")
s, t_pred = timed(lambda X: xgbm.predict_proba(X)[:, 1], X_test)
show(evaluate("XGBoost", y_test, s, 0.5, t_fit, t_pred))

## 8. Model 4: Isolation Forest (unsupervised)
Isolation Forest is trained on **benign training flows only**. It never sees an attack label, which models a SOC that has plenty of normal traffic but no labelled examples of a new attack. It outputs an anomaly score rather than a probability, so its alert threshold is chosen on the validation set (the threshold with the highest F1). This choice uses validation labels and is reported as such.

In [ ]:
iso = IsolationForest(n_estimators=200, max_samples=256, contamination="auto",
                      n_jobs=N_JOBS, random_state=RANDOM_STATE)
_, t_fit = timed(iso.fit, X_train[y_train == 0])

iso_val = -iso.score_samples(X_val)              # higher score = more anomalous
p, r, th = precision_recall_curve(y_val, iso_val)
f1s = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-12, None)
iso_threshold = float(th[np.argmax(f1s)])
print(f"Isolation Forest threshold chosen on validation: {iso_threshold:.4f}")

s, t_pred = timed(lambda X: -iso.score_samples(X), X_test)
show(evaluate("Isolation Forest", y_test, s, iso_threshold, t_fit, t_pred))

## 9. Comparison of models on the test set

In [ ]:
res = pd.DataFrame(results).set_index("model")
res.to_csv(f"{OUT_DIR}/02_model_comparison.csv")
cols = ["accuracy", "precision", "recall_detection_rate", "f1", "fpr", "fnr", "mcc", "pr_auc",
        "false_alarms_per_100k_benign", "fit_time_s"]
display(res[cols].round(4))

fig, axes = plt.subplots(1, len(res), figsize=(4 * len(res), 3.8))
for ax, (name, row) in zip(np.atleast_1d(axes), res.iterrows()):
    cm = np.array([[row.TN, row.FP], [row.FN, row.TP]])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{int(cm[i, j]):,}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["Benign", "Attack"]); ax.set_yticks([0, 1], ["Benign", "Attack"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(name)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_confusion_matrices.png", dpi=200)
plt.show()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
for name, sc in test_scores.items():
    fpr_c, tpr_c, _ = roc_curve(y_test, sc)
    a1.plot(fpr_c, tpr_c, label=name)
    p_c, r_c, _ = precision_recall_curve(y_test, sc)
    a2.plot(r_c, p_c, label=name)
a1.set_xscale("log"); a1.set_xlim(1e-5, 1)
a1.set_xlabel("False positive rate (log scale)"); a1.set_ylabel("Detection rate"); a1.set_title("ROC curves")
a2.set_xlabel("Recall (detection rate)"); a2.set_ylabel("Precision"); a2.set_title("Precision-recall curves")
a1.legend(); a2.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_roc_pr_curves.png", dpi=200)
plt.show()

## 10. Detection rate per attack type
An overall detection rate can hide weak performance on rare attacks. This table shows, for each original label in the test set, the share flagged as an attack. For BENIGN the value is the share wrongly flagged, i.e. the false positive rate.

In [ ]:
per_label = {}
for name, sc in test_scores.items():
    flagged = sc >= thresholds_used[name]
    per_label[name] = pd.Series(flagged).groupby(labels_test).mean()
per_label = pd.DataFrame(per_label)
per_label.insert(0, "test_rows", pd.Series(labels_test).value_counts())
per_label = per_label.sort_values("test_rows", ascending=False)
per_label.to_csv(f"{OUT_DIR}/02_detection_rate_by_label.csv")
display(per_label.round(4))
print("Note: classes with very few test rows (e.g. Heartbleed, Infiltration, SQL injection) give unstable "
      "rates; one flow changes the result by a large amount.")

## 11. Threshold trade-off (best supervised model)
A 0.5 threshold is a default, not a SOC decision. This table, built on the **validation** set, shows how detection and false alarms change with the threshold, so TeleComNet could choose a threshold that matches analyst capacity.

In [ ]:
sup = res.drop(index="Isolation Forest", errors="ignore")
best_name = sup["mcc"].idxmax()
best_model = {"Logistic Regression": logreg, "Random Forest": rf, "XGBoost": xgbm}[best_name]
print("Best supervised model by MCC on the test set:", best_name)

val_score = best_model.predict_proba(X_val)[:, 1]
rows = []
for t in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    r_ = evaluate(best_name, y_val, val_score, t, store=False)
    rows.append({k: r_[k] for k in ["threshold", "precision", "recall_detection_rate", "f1", "fpr", "fnr",
                                     "false_alarms_per_100k_benign", "FP", "FN"]})
thr = pd.DataFrame(rows)
thr.to_csv(f"{OUT_DIR}/02_threshold_tradeoff_validation.csv", index=False)
display(thr.round(5))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(thr.threshold, thr.recall_detection_rate, marker="o", label="Detection rate")
ax.plot(thr.threshold, thr.precision, marker="o", label="Precision")
ax2 = ax.twinx()
ax2.plot(thr.threshold, thr.false_alarms_per_100k_benign, marker="s", color="grey", label="False alarms per 100k benign")
ax.set_xlabel("Alert threshold"); ax.set_ylabel("Rate"); ax2.set_ylabel("False alarms per 100k benign flows")
ax.set_title(f"{best_name}: threshold trade-off (validation set)")
ax.legend(loc="lower left"); ax2.legend(loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_threshold_tradeoff.png", dpi=200)
plt.show()

## 12. Explainability: which features drive the decisions?
Built-in importances from tree models can be biased towards features with many distinct values, so permutation importance (the drop in F1 when a feature is shuffled) is also computed for the best model on a sample of the test set.

In [ ]:
imp = pd.DataFrame({"feature": feature_cols,
                    "rf_impurity": rf.feature_importances_,
                    "xgb_gain_share": xgbm.feature_importances_})

n_perm = min(50_000, len(X_test))
rng = np.random.default_rng(RANDOM_STATE)
pi_idx = rng.choice(len(X_test), n_perm, replace=False)
pi = permutation_importance(best_model, X_test[pi_idx], y_test[pi_idx], scoring="f1", n_repeats=3,
                            n_jobs=1, random_state=RANDOM_STATE)
imp[f"permutation_{best_name}"] = pi.importances_mean
imp = imp.sort_values(f"permutation_{best_name}", ascending=False)
imp.to_csv(f"{OUT_DIR}/02_feature_importance.csv", index=False)
display(imp.head(15).round(4))

top = imp.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.feature, top[f"permutation_{best_name}"])
ax.set_xlabel("Mean drop in F1 when shuffled")
ax.set_title(f"Top 15 features by permutation importance ({best_name})")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_feature_importance.png", dpi=200)
plt.show()

## 13. Experiment: does Destination Port act as a shortcut?
XGBoost is retrained with Destination Port added back, using the same number of trees. If performance rises noticeably, part of the gain may come from lab-specific port patterns rather than traffic behaviour, which supports excluding it.

In [ ]:
if RUN_PORT_ABLATION and PORT_COL in df.columns:
    port = df[PORT_COL].to_numpy(dtype=np.float32)
    Xp_train = np.column_stack([X_train, port[idx_train]])
    Xp_test = np.column_stack([X_test, port[idx_test]])
    params = {**xgb_params, "n_estimators": best_n_trees}
    m_port = XGBClassifier(**params).fit(Xp_train, y_train, verbose=False)
    with_port = evaluate("XGBoost + Destination Port", y_test, m_port.predict_proba(Xp_test)[:, 1], 0.5, store=False)
    without = res.loc["XGBoost"].to_dict(); without["model"] = "XGBoost (no port)"
    ablation = pd.DataFrame([without, with_port]).set_index("model")[
        ["precision", "recall_detection_rate", "f1", "fpr", "fnr", "mcc"]]
    ablation.to_csv(f"{OUT_DIR}/02_port_ablation.csv")
    display(ablation.round(5))
    port_rank = pd.Series(m_port.feature_importances_, index=feature_cols + [PORT_COL]).rank(ascending=False)
    print(f"Destination Port importance rank when included: {int(port_rank[PORT_COL])} of {len(port_rank)}")
    del Xp_train, Xp_test

## 14. Experiment: detecting attack types never seen in training
A supervised model only learns the attacks in its training data. Here each attack family is removed from training in turn, XGBoost is retrained, and its detection rate on that family in the test set is measured. Isolation Forest is shown alongside: it never saw any attacks, so its figures come from the model in Section 8. This tests the claim that anomaly detection helps against new (zero-day style) attacks.

In [ ]:
if RUN_UNSEEN_ATTACKS:
    fam_all = df["attack_family"].to_numpy()
    fam_train, fam_test = fam_all[idx_train], fam_all[idx_test]
    iso_flag = test_scores["Isolation Forest"] >= thresholds_used["Isolation Forest"]
    xgb_flag_full = test_scores["XGBoost"] >= 0.5
    rows = []
    for fam in [f for f in pd.Series(fam_test).value_counts().index if f != "Benign"]:
        if (fam_test == fam).sum() < 20:
            continue                      # too few test rows to give a meaningful rate
        keep = fam_train != fam
        m = XGBClassifier(**{**xgb_params, "n_estimators": best_n_trees})
        m.fit(X_train[keep], y_train[keep], verbose=False)
        mask = fam_test == fam
        unseen_flag = m.predict_proba(X_test[mask])[:, 1] >= 0.5
        benign_mask = fam_test == "Benign"
        rows.append({"held_out_family": fam, "test_rows": int(mask.sum()),
                     "xgb_seen_detection": xgb_flag_full[mask].mean(),
                     "xgb_unseen_detection": unseen_flag.mean(),
                     "isolation_forest_detection": iso_flag[mask].mean(),
                     "xgb_unseen_benign_fpr": (m.predict_proba(X_test[benign_mask])[:, 1] >= 0.5).mean()})
        print(f"done: {fam}")
    unseen = pd.DataFrame(rows).set_index("held_out_family")
    unseen.to_csv(f"{OUT_DIR}/02_unseen_attack_experiment.csv")
    display(unseen.round(4))

## 15. Extension: multi-class classification by attack family
A SOC needs to know not just whether a flow is malicious but what kind of attack it is, because the response differs. XGBoost is trained to predict the attack family, with balanced sample weights. Macro-averaged F1 is reported because it gives every family equal weight, however rare.

In [ ]:
if RUN_MULTICLASS:
    fam_codes, fam_names = pd.factorize(df["attack_family"], sort=True)
    yf_train, yf_val, yf_test = fam_codes[idx_train], fam_codes[idx_val], fam_codes[idx_test]
    mc = XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=8, subsample=0.8, colsample_bytree=0.8,
                       tree_method="hist", objective="multi:softprob", eval_metric="mlogloss",
                       early_stopping_rounds=30, n_jobs=N_JOBS, random_state=RANDOM_STATE)
    _, t_fit = timed(mc.fit, X_train, yf_train, sample_weight=compute_sample_weight("balanced", yf_train),
                     eval_set=[(X_val, yf_val)], verbose=False)
    yf_pred = mc.predict(X_test)
    report = pd.DataFrame(classification_report(yf_test, yf_pred, labels=range(len(fam_names)),
                                                target_names=list(fam_names), output_dict=True,
                                                zero_division=0)).T
    report.to_csv(f"{OUT_DIR}/02_multiclass_report.csv")
    display(report.round(4))
    print(f"Macro F1: {report.loc['macro avg', 'f1-score']:.4f} | fit time {t_fit:.0f}s")

    cmf = confusion_matrix(yf_test, yf_pred, labels=range(len(fam_names)))
    cmn = cmf / cmf.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(fam_names)):
        for j in range(len(fam_names)):
            if cmf[i, j]:
                ax.text(j, i, f"{cmf[i, j]:,}", ha="center", va="center", fontsize=7,
                        color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xticks(range(len(fam_names)), fam_names, rotation=45, ha="right")
    ax.set_yticks(range(len(fam_names)), fam_names)
    ax.set_xlabel("Predicted family"); ax.set_ylabel("Actual family")
    ax.set_title("Multi-class XGBoost (colour = row share, numbers = counts)")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/02_multiclass_confusion_matrix.png", dpi=200)
    plt.show()

## 16. Sample predictions, saved models and run configuration

In [ ]:
# Sample predictions: a few test flows of each type, with every model's score and decision
rng = np.random.default_rng(RANDOM_STATE)
picks = []
for lbl in pd.unique(labels_test):
    pos = np.where(labels_test == lbl)[0]
    picks.extend(rng.choice(pos, min(3, len(pos)), replace=False))
picks = np.array(picks)
sample = pd.DataFrame({"true_label": labels_test[picks], "is_attack": y_test[picks]})
for name, sc in test_scores.items():
    sample[f"{name} score"] = sc[picks].round(4)
    sample[f"{name} alert"] = (sc[picks] >= thresholds_used[name]).astype(int)
sample.to_csv(f"{OUT_DIR}/02_sample_predictions.csv", index=False)
display(sample.head(20))

# Saved for reuse in CW2. The models folder is excluded from GitHub (.gitignore), because a
# Random Forest file can be hundreds of MB.
joblib.dump({"model": best_model, "features": feature_cols, "threshold": 0.5}, f"{MODEL_DIR}/best_binary_model.joblib")
joblib.dump({"model": iso, "features": feature_cols, "threshold": iso_threshold}, f"{MODEL_DIR}/isolation_forest.joblib")

with open(f"{OUT_DIR}/02_run_config.json", "w") as fh:
    json.dump({"rows": int(len(df)), "sample_frac": SAMPLE_FRAC, "n_features": len(feature_cols),
               "split": "stratified by Label, 70/15/15", "random_state": RANDOM_STATE,
               "logistic_regression": {"max_iter": 1000, "class_weight": "balanced", "scaling": "StandardScaler"},
               "random_forest": {"n_estimators": 100, "class_weight": "balanced_subsample"},
               "xgboost": {**{k: v for k, v in xgb_params.items() if k != "n_jobs"}, "best_n_trees": int(best_n_trees)},
               "isolation_forest": {"n_estimators": 200, "max_samples": 256, "trained_on": "benign training flows",
                                    "threshold": iso_threshold, "threshold_rule": "max F1 on validation"},
               "best_supervised_model": best_name,
               "versions": {"sklearn": sklearn.__version__, "xgboost": xgb.__version__, "pandas": pd.__version__}},
              fh, indent=2, default=float)
print("Saved models and outputs/02_run_config.json")

## Notes for the report (fill in from your own outputs)
- Model comparison table: `outputs/02_model_comparison.csv` and `02_confusion_matrices.png`
- Which model is best, and by which metric? Why MCC, F1 and PR-AUC rather than accuracy?
- Rare attacks: `outputs/02_detection_rate_by_label.csv`. Which attack types are missed?
- Operational trade-off: `outputs/02_threshold_tradeoff_validation.csv`. What threshold would you recommend for TeleComNet, and why?
- Explainability: `outputs/02_feature_importance.csv`. Do the top features make sense for these attacks?
- Destination Port: `outputs/02_port_ablation.csv`. Does including it change results?
- Unseen attacks: `outputs/02_unseen_attack_experiment.csv`. Does Isolation Forest help when the supervised model has never seen an attack?
- Multi-class: `outputs/02_multiclass_report.csv`. Which families are confused with each other?
- Limitations: CICIDS2017 is a lab dataset from 2017; random splits keep flows from the same attack session in both train and test; hyperparameters were not extensively tuned.